# Лабораторная работа №8  
# Оптимизация и деплой LLM

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить методы оптимизации больших языковых моделей
- Освоить квантование моделей (4-bit, 8-bit)
- Научиться развертывать модели через FastAPI
- Познакомиться с контейнеризацией через Docker
- Сравнить производительность до и после оптимизации

---

## 2. Теоретические сведения

### 2.1. Проблемы деплоя LLM

| Проблема | Описание | Решение |
|----------|----------|--------|
| Большой размер | Модели занимают ГБ памяти | Квантование, прунинг |
| Медленный инференс | Высокая задержка генерации | vLLM, TensorRT |
| Высокая стоимость | Дорогие GPU для продакшена | Оптимизация, дистилляция |

### 2.2. Квантование

**Квантование** — уменьшение битности весов модели:
- FP32 → INT8 (4x экономия памяти)
- FP32 → INT4 (8x экономия памяти)
- QLoRA: квантование + LoRA для fine-tuning

### 2.3. Фреймворки для инференса

| Фреймворк | Особенности | Применение |
|-----------|-------------|------------|
| vLLM | PagedAttention, высокая скорость | Продакшен |
| TGI (Text Generation Inference) | Оптимизирован для Hugging Face | HF Hub |
| llama.cpp | CPU-инференс, GGUF формат | Локальный запуск |

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Загрузите модель с 4-bit квантованием через bitsandbytes
2. Сравните размер модели до и после квантования
3. Создайте простой API на FastAPI для генерации текста
4. Протестируйте время генерации
5. Оцените качество ответов

### 3.2. Продвинутый уровень (дополнительно)

- Реализуйте endpoint для стриминга токенов
- Добавьте кэширование частых запросов
- Сравните скорость vLLM и transformers

### 3.3. Индивидуальное задание

Подготовьте Dockerfile для деплоя модели:
- **Вариант A:** Контейнер с FastAPI + quantized модель
- **Вариант B:** Контейнер с vLLM сервером

Обоснуйте выбор конфигурации.

---

## 4. Ход работы

### 4.1. Подготовка окружения

In [ ]:
# Установка зависимостей
!pip install transformers torch accelerate bitsandbytes -q
!pip install fastapi uvicorn pydantic -q

# Проверка версий
import transformers
import torch

print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU доступен: {torch.cuda.is_available()}")

### 4.2. Загрузка модели с квантованием

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Модель для демонстрации (небольшая)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Загрузка модели с 4-bit квантованием...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Модель загружена!")

### 4.3. Оценка размера модели

In [ ]:
# Подсчет параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {total_params:,}")

# Примерный размер в памяти (4-bit)
size_4bit = total_params * 4 / 8 / 1024 / 1024  # MB
size_16bit = total_params * 16 / 8 / 1024 / 1024  # MB

print(f"\nРазмер модели:")
print(f"  4-bit: {size_4bit:.2f} MB")
print(f"  16-bit: {size_16bit:.2f} MB")
print(f"  Экономия: {(1 - size_4bit/size_16bit)*100:.1f}%")

### 4.4. Тестирование генерации

In [ ]:
import time

def generate_text(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    start_time = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    elapsed = time.time() - start_time
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    tokens_per_sec = max_tokens / elapsed if elapsed > 0 else 0
    
    return response, elapsed, tokens_per_sec

# Тестовые запросы
prompts = [
    "Расскажи кратко о Python программировании",
    "Какие бывают типы данных в Python?",
    "Объясни что такое функция простыми словами"
]

for prompt in prompts:
    print(f"\nЗапрос: {prompt}")
    response, elapsed, tps = generate_text(prompt, max_tokens=50)
    print(f"Ответ: {response[:200]}...")
    print(f"Время: {elapsed:.2f} сек, Токенов/сек: {tps:.1f}")

### 4.5. Создание FastAPI приложения

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from threading import Thread

# Определение схем данных
class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 100
    temperature: float = 0.7

class GenerateResponse(BaseModel):
    text: str
    tokens_per_sec: float
    elapsed_time: float

# Создание приложения
app = FastAPI(title="LLM API", description="API для генерации текста")

@app.post("/generate", response_model=GenerateResponse)
async def generate(request: GenerateRequest):
    inputs = tokenizer(request.prompt, return_tensors="pt").to(model.device)
    
    start_time = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=request.max_tokens,
        temperature=request.temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    elapsed = time.time() - start_time
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    tps = request.max_tokens / elapsed if elapsed > 0 else 0
    
    return GenerateResponse(text=text, tokens_per_sec=tps, elapsed_time=elapsed)

@app.get("/")
async def root():
    return {"message": "LLM API готов! Используйте POST /generate"}

print("FastAPI приложение создано!")
print("Для запуска используйте: uvicorn app:app --host 0.0.0.0 --port 8000")

### 4.6. Пример Dockerfile

In [ ]:
dockerfile_content = '''
FROM python:3.10-slim

WORKDIR /app

# Установка системных зависимостей
RUN apt-get update && apt-get install -y git curl

# Установка Python зависимостей
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Копирование кода
COPY app.py .

# Экспорт порта
EXPOSE 8000

# Запуск приложения
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements_content = '''
transformers>=4.40.0
torch>=2.0.0
accelerate>=0.25.0
bitsandbytes>=0.42.0
fastapi>=0.109.0
uvicorn>=0.27.0
pydantic>=2.0.0
'''

print("=== Dockerfile ===")
print(dockerfile_content)
print("\n=== requirements.txt ===")
print(requirements_content)

---

## 5. Контрольные вопросы

1. Какие преимущества дает квантование моделей?
2. В чем разница между 4-bit и 8-bit квантованием?
3. Почему vLLM быстрее стандартного transformers?
4. Какие проблемы могут возникнуть при деплое LLM?
5. Как масштабировать сервис для обработки множества запросов?

---

## 6. Выводы

В ходе работы были изучены методы оптимизации LLM, реализовано 4-bit квантование и создан API для генерации текста с использованием FastAPI.

---